# Aula 07 | RAG: documentos como fonte de evidência

**Inteligência Artificial para Decisão** · Finanças · notebook local.

**Pergunta:** Como responder a uma pergunta sobre relatórios trimestrais citando o trecho correto?

**Procedimento:** execute as células, confira os resultados e modifique uma hipótese. Não há chamadas a serviços externos. Todos os dados são sintéticos ou fictícios.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Localiza a raiz do curso, inclusive ao abrir o notebook a partir da pasta da aula.
def achar_raiz():
    origem = Path.cwd().resolve()
    for pasta in (origem, *origem.parents):
        if (pasta / "dados" / "README.md").exists() and (pasta / "src").exists():
            return pasta
    raise FileNotFoundError("Abra o notebook dentro do repositório extraído, preservando dados/ e src/.")

ROOT = achar_raiz()
sys.path.insert(0, str(ROOT / "src"))
print("Repositório:", ROOT)


## Corpus e cortes por título

In [ ]:
from iad.retrieval import load_corpus,chunk_headings,TfidfIndex,BM25Index
corpus=load_corpus(ROOT/'dados/relatorios')
corpus.update(load_corpus(ROOT/'dados/faq'))
chunks=chunk_headings(corpus,max_chars=900)
print('Documentos:',len(corpus),'Trechos:',len(chunks))
print(chunks[0])

## Recuperação TF-IDF e BM25

In [ ]:
tf=TfidfIndex(chunks)
bm=BM25Index(chunks)
pergunta='receita líquida da Horizonte no segundo trimestre de 2026'
for name,index in [('TF-IDF',tf),('BM25',bm)]:
    print(name,[(h.doc_id,h.section,round(h.score,3)) for h in index.search(pergunta,k=3)])

## Regra de citação

In [ ]:
fontes={h.doc_id for h in tf.search(pergunta,k=3)}
print('Documentos recuperados:',fontes)
print('Um trecho não encontrado não autoriza o sistema a inventar um número.')

## Síntese individual

1. Qual pergunta este experimento responde e qual **não** responde?
2. Que valores foram calculados e quais foram apenas interpretados?
3. Que alteração você realizou? Registre a saída e explique sua escolha.
4. Qual erro seria especialmente relevante para uma decisão em **Finanças**?

**Entrega:** Protótipo de RAG sobre documentos fictícios com trechos identificados.